In [ ]:
from abc import ABC, abstractmethod
from typing import Tuple

import torch
from pydantic import BaseModel, ConfigDict, field_validator
from torch import Tensor


class BaseTarget(BaseModel, ABC):
    """Base class for all target types with Pydantic validation."""

    model_config = ConfigDict(arbitrary_types_allowed=True)

    @abstractmethod
    def select_from_output(self, output: Tensor) -> Tensor:
        """Select the appropriate part of the model output."""
        pass

    @abstractmethod
    def is_multi_target(self) -> bool:
        """Whether this represents multiple targets."""
        pass


class NoTarget(BaseTarget):
    """No target selection - return the full model output."""

    def select_from_output(self, output: Tensor) -> Tensor:
        return output

    def is_multi_target(self) -> bool:
        return False


class SingleTargetAcrossBatch(BaseTarget):
    """Single target index applied to all examples in the batch."""

    index: int

    @field_validator("index")
    @classmethod
    def validate_index(cls, v):
        if v < 0:
            raise ValueError("Target index must be non-negative")
        return v

    def select_from_output(self, output: Tensor) -> Tensor:
        """Select output[:, self.index] or output[..., self.index]"""
        if self.index >= output.shape[-1]:
            raise ValueError(
                f"Target index {self.index} >= output size {output.shape[-1]}"
            )
        return output[..., self.index]

    def is_multi_target(self) -> bool:
        return False


class MultiIndexTargetAcrossBatch(BaseTarget):
    """Multiple indices for selecting nested dimensions."""

    indices: Tuple[int, ...]

    @field_validator("indices")
    @classmethod
    def validate_indices(cls, v):
        if len(v) == 0:
            raise ValueError("Must provide at least one index")
        for idx in v:
            if idx < 0:
                raise ValueError("All indices must be non-negative")
        return v

    def select_from_output(self, output: Tensor) -> Tensor:
        """Select output[:, indices[0], indices[1], ...]"""
        if len(self.indices) >= len(output.shape):
            raise ValueError(
                f"Too many indices {len(self.indices)} for output shape {output.shape}"
            )

        selection = (slice(None),) + self.indices
        return output[selection]

    def is_multi_target(self) -> bool:
        return False


class PerExampleTarget(BaseTarget):
    """Different target index for each example in the batch."""

    targets: Union[List[int], Tensor]

    @field_validator("targets")
    @classmethod
    def validate_targets(cls, v):
        if isinstance(v, list):
            if len(v) == 0:
                raise ValueError("Target list cannot be empty")
            for target in v:
                if not isinstance(target, int) or target < 0:
                    raise ValueError("All targets must be non-negative integers")
            return torch.tensor(v)
        elif isinstance(v, Tensor):
            if v.dim() != 1:
                raise ValueError("Target tensor must be 1-dimensional")
            if torch.any(v < 0):
                raise ValueError("All targets must be non-negative")
            return v
        else:
            raise ValueError("Targets must be a list of integers or 1D tensor")

    def select_from_output(self, output: Tensor) -> Tensor:
        """Use torch.gather to select different index per example."""
        batch_size = output.shape[0]

        if self.targets.shape[0] != batch_size:
            raise ValueError(
                f"Target length {self.targets.shape[0]} != batch size {batch_size}"
            )

        if len(output.shape) != 2:
            raise ValueError(f"PerExampleTarget requires 2D output, got {output.shape}")

        if torch.any(self.targets >= output.shape[1]):
            raise ValueError(f"Target indices must be < {output.shape[1]}")

        return torch.gather(
            output, 1, self.targets.unsqueeze(1).to(output.device)
        ).squeeze(1)

    def is_multi_target(self) -> bool:
        return len(torch.unique(self.targets)) > 1


class PerExampleMultiIndexTarget(BaseTarget):
    """Different multi-index target for each example in the batch."""

    targets: List[Tuple[int, ...]]

    @field_validator("targets")
    @classmethod
    def validate_targets(cls, v):
        if len(v) == 0:
            raise ValueError("Target list cannot be empty")

        for i, target_tuple in enumerate(v):
            if not isinstance(target_tuple, tuple):
                raise ValueError(f"Target {i} must be a tuple")
            if len(target_tuple) == 0:
                raise ValueError(f"Target tuple {i} cannot be empty")
            for j, idx in enumerate(target_tuple):
                if not isinstance(idx, int) or idx < 0:
                    raise ValueError(
                        f"Target {i}, index {j} must be a non-negative integer"
                    )

        return v

    def select_from_output(self, output: Tensor) -> Tensor:
        """Select different multi-index for each example."""
        batch_size = output.shape[0]

        if len(self.targets) != batch_size:
            raise ValueError(
                f"Target length {len(self.targets)} != batch size {batch_size}"
            )

        results = []
        for i, indices in enumerate(self.targets):
            if len(indices) + 1 >= len(output.shape):
                raise ValueError(
                    f"Too many indices {len(indices)} for output shape {output.shape}"
                )

            selection = (i,) + indices
            results.append(output[selection])

        return torch.stack(results)

    def is_multi_target(self) -> bool:
        return len(set(self.targets)) > 1


# Factory function for backward compatibility (if needed)
def create_target(raw_target) -> BaseTarget:
    """Convert old-style targets to new clear target classes with validation."""
    if raw_target is None:
        return NoTarget()

    elif isinstance(raw_target, int):
        return SingleTarget(index=raw_target)

    elif isinstance(raw_target, torch.Tensor):
        if torch.numel(raw_target) == 1:
            return SingleTarget(index=int(raw_target.item()))
        else:
            return PerExampleTarget(targets=raw_target)

    elif isinstance(raw_target, tuple):
        return MultiIndexTarget(indices=raw_target)

    elif isinstance(raw_target, list):
        if len(raw_target) == 0:
            raise ValueError("Empty target list")

        if isinstance(raw_target[0], int):
            return PerExampleTarget(targets=raw_target)
        elif isinstance(raw_target[0], tuple):
            return PerExampleMultiIndexTarget(targets=raw_target)
        else:
            raise ValueError(f"Unsupported list element type: {type(raw_target[0])}")

    else:
        raise ValueError(f"Unsupported target type: {type(raw_target)}")

In [ ]:
# Test Pydantic validation - now everything gets validated!
import torch

print("Testing Pydantic validation:")
print("=" * 30)

# Valid cases
try:
    target1 = SingleTarget(index=5)
    print(f"✅ SingleTarget(index=5): {target1.index}")
except Exception as e:
    print(f"❌ SingleTarget(index=5): {e}")

try:
    target2 = PerExampleTarget(targets=[0, 2, 1])
    print(f"✅ PerExampleTarget(targets=[0,2,1]): shape {target2.targets.shape}")
except Exception as e:
    print(f"❌ PerExampleTarget(targets=[0,2,1]): {e}")

try:
    target3 = MultiIndexTarget(indices=(1, 2, 0))
    print(f"✅ MultiIndexTarget(indices=(1,2,0)): {target3.indices}")
except Exception as e:
    print(f"❌ MultiIndexTarget(indices=(1,2,0)): {e}")

print("\nTesting validation errors:")
print("=" * 30)

# Invalid cases - these should raise validation errors
try:
    bad_target1 = SingleTarget(index=-1)  # Negative index
    print(f"❌ Should have failed: {bad_target1}")
except Exception as e:
    print(f"✅ Caught negative index: {e}")

try:
    bad_target2 = PerExampleTarget(targets=[])  # Empty list
    print(f"❌ Should have failed: {bad_target2}")
except Exception as e:
    print(f"✅ Caught empty list: {e}")

try:
    bad_target3 = PerExampleTarget(targets=[1, -2, 3])  # Negative in list
    print(f"❌ Should have failed: {bad_target3}")
except Exception as e:
    print(f"✅ Caught negative in list: {e}")

try:
    bad_target4 = MultiIndexTarget(indices=())  # Empty tuple
    print(f"❌ Should have failed: {bad_target4}")
except Exception as e:
    print(f"✅ Caught empty indices: {e}")

try:
    bad_target5 = PerExampleMultiIndexTarget(targets=[])  # Empty list
    print(f"❌ Should have failed: {bad_target5}")
except Exception as e:
    print(f"✅ Caught empty multi-index list: {e}")

In [ ]:
# Test runtime validation with actual tensors
batch_size = 4
num_classes = 10
output = torch.randn(batch_size, num_classes)

print("Testing runtime validation with tensors:")
print("=" * 40)

# Valid operations
try:
    target1 = SingleTarget(index=5)  # Valid index
    result1 = target1.select_from_output(output)
    print(f"✅ SingleTarget(5) with output shape {output.shape}: result shape {result1.shape}")
except Exception as e:
    print(f"❌ SingleTarget(5): {e}")

try:
    target2 = PerExampleTarget(targets=[0, 2, 5, 1])  # Valid per-example targets
    result2 = target2.select_from_output(output)
    print(f"✅ PerExampleTarget([0,2,5,1]): result shape {result2.shape}")
except Exception as e:
    print(f"❌ PerExampleTarget([0,2,5,1]): {e}")

print("\nTesting runtime errors:")
print("=" * 25)

# Runtime validation errors
try:
    target3 = SingleTarget(index=15)  # Index too large for output
    result3 = target3.select_from_output(output)
    print(f"❌ Should have failed: {result3.shape}")
except Exception as e:
    print(f"✅ Caught index too large: {e}")

try:
    target4 = PerExampleTarget(targets=[0, 1])  # Wrong batch size
    result4 = target4.select_from_output(output)
    print(f"❌ Should have failed: {result4.shape}")
except Exception as e:
    print(f"✅ Caught batch size mismatch: {e}")

try:
    target5 = PerExampleTarget(targets=[0, 2, 15, 1])  # Index too large
    result5 = target5.select_from_output(output)
    print(f"❌ Should have failed: {result5.shape}")
except Exception as e:
    print(f"✅ Caught target index too large: {e}")

In [ ]:
# Usage in explainer is now super obvious
class ClearExplainer:
    """Explainer with crystal clear target usage."""
    
    def __init__(self, model):
        self.model = model
    
    def explain_single_target(self, inputs, target_index: int):
        """Explain with same target for all examples."""
        target = SingleTarget(target_index)
        output = self.model(inputs)
        return target.select_from_output(output)
    
    def explain_per_example(self, inputs, target_indices: List[int]):
        """Explain with different target per example."""
        target = PerExampleTarget(target_indices)
        output = self.model(inputs)
        return target.select_from_output(output)
    
    def explain_no_target(self, inputs):
        """Explain full model output."""
        target = NoTarget()
        output = self.model(inputs)
        return target.select_from_output(output)
    
    def explain_multi_index(self, inputs, *indices: int):
        """Explain with multi-dimensional target selection."""
        target = MultiIndexTarget(*indices)
        output = self.model(inputs)
        return target.select_from_output(output)


# Test the clear explainer
model = torch.nn.Linear(10, 5)
explainer = ClearExplainer(model)
test_input = torch.randn(2, 10)

print("\nClear explainer usage:")
print("=" * 25)

# Crystal clear what each method does!
result1 = explainer.explain_single_target(test_input, target_index=2)
print(f"Single target (index 2): shape {result1.shape}")

result2 = explainer.explain_per_example(test_input, target_indices=[0, 4])
print(f"Per-example targets [0,4]: shape {result2.shape}")

result3 = explainer.explain_no_target(test_input)
print(f"No target selection: shape {result3.shape}")

In [ ]:
# Modern explainer with Pydantic validation
class ValidatedExplainer:
    """Explainer with full Pydantic validation on targets."""
    
    def __init__(self, model):
        self.model = model
    
    def explain(self, inputs, target: BaseTarget):
        """
        Now targets are validated at creation time AND runtime!
        """
        # Target is already validated by Pydantic
        output = self.model(inputs)
        
        # Runtime validation happens in select_from_output
        selected_output = target.select_from_output(output)
        
        return {
            'target_type': type(target).__name__,
            'target_data': target.model_dump(),  # Pydantic serialization
            'is_multi_target': target.is_multi_target(),
            'output_shape': output.shape,
            'selected_shape': selected_output.shape,
        }


# Test with validation
model = torch.nn.Linear(10, 5)
explainer = ValidatedExplainer(model)
test_input = torch.randn(3, 10)

print("Validated explainer usage:")
print("=" * 30)

# Valid usage
try:
    target1 = SingleTarget(index=2)
    result1 = explainer.explain(test_input, target1)
    print(f"✅ {result1['target_type']}: {result1['output_shape']} -> {result1['selected_shape']}")
    print(f"   Target data: {result1['target_data']}")
except Exception as e:
    print(f"❌ Error: {e}")

# Invalid usage - caught at creation time
try:
    bad_target = SingleTarget(index=-1)  # This will fail immediately
    result2 = explainer.explain(test_input, bad_target)
except Exception as e:
    print(f"✅ Validation caught bad target: {e}")

print(f"\n🎉 NO MORE SILENT FAILURES - Everything is validated!")